# HumanAI Detect - Kisa-Metin GPT-4o-mini Takviyesi: On Isleme (Asama 2, Colab GPU)

**Amac:** `ai_raw_openai_short` (300 ornek) ve `ai_humanized_openai_short` (300 ornek) havuzlarini Stanza dilbilimsel analiz + BERTurk/DistilBERT capraz-perplexity + GPT2-Turkish token-rank + burstiness ile isler. Diger short etiketlerle (human_short/ai_raw_short/ai_humanized_short) AYNI kisa kriterle (--min-tokens 5 --max-tokens 30) calistirilir -- ana TOPUP_LABELS'in (30/850) kriteriyle KARISTIRILMAMALI.

**Neden Colab:** Yerel CPU'da BERT-tabanli pseudo-perplexity ~200sn/ornek surer.

**ONEMLI -- once kontrol et:** Yerel kod degisikliklerinin GitHub'a push edildiginden emin ol.


In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')


In [ ]:
# 2. Repo klonla
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -3


In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q


In [ ]:
# 4. Stanza Turkce modelini indir
import stanza
stanza.download('tr')


In [ ]:
# 5. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')


In [ ]:
# 6. Kaynak dosyalari yerel makineden yukle: ai_raw_openai_short.jsonl VE ai_humanized_openai_short.jsonl (ikisini birden secebilirsin)
from pathlib import Path
from google.colab import files

Path('data/raw/ai_raw_openai_short').mkdir(parents=True, exist_ok=True)
Path('data/raw/ai_humanized_openai_short').mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized_openai_short' in name:
        Path(name).rename('data/raw/ai_humanized_openai_short/ai_humanized_openai_short.jsonl')
    elif 'ai_raw_openai_short' in name:
        Path(name).rename('data/raw/ai_raw_openai_short/ai_raw_openai_short.jsonl')


In [ ]:
# 6b. (SADECE onceki bir Colab oturumundan kaldiysa) kismi interim checkpoint'lerini geri yukle -- ilk kez calistiriyorsan bu hucreyi ATLA
from pathlib import Path
from google.colab import files

Path('data/interim/ai_raw_openai_short').mkdir(parents=True, exist_ok=True)
Path('data/interim/ai_humanized_openai_short').mkdir(parents=True, exist_ok=True)

print('Onceki oturumdan kalan interim dosyalarini sec, yoksa bu hucreyi atla')
uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized_openai_short' in name:
        Path(name).rename('data/interim/ai_humanized_openai_short/ai_humanized_openai_short.jsonl')
    elif 'ai_raw_openai_short' in name:
        Path(name).rename('data/interim/ai_raw_openai_short/ai_raw_openai_short.jsonl')


In [ ]:
# 7a. ai_raw_openai_short on isleme -- KISA kriter (5-30 kelime) ZORUNLU, ana TOPUP kriterini (30-850) KULLANMA
!python scripts/preprocess.py --label ai_raw_openai_short --min-tokens 5 --max-tokens 30


In [ ]:
# 7b. Guvenlik: hemen indir
from pathlib import Path
from google.colab import files

src = Path('data/interim/ai_raw_openai_short/ai_raw_openai_short.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')


In [ ]:
# 8a. ai_humanized_openai_short on isleme -- KISA kriter
!python scripts/preprocess.py --label ai_humanized_openai_short --min-tokens 5 --max-tokens 30


In [ ]:
# 8b. Sonucu indir
from pathlib import Path
from google.colab import files

src = Path('data/interim/ai_humanized_openai_short/ai_humanized_openai_short.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')


## Yerel Makineye Aktarim

Indirilen iki dosyayi:
1. `data/interim/ai_raw_openai_short/ai_raw_openai_short.jsonl`
2. `data/interim/ai_humanized_openai_short/ai_humanized_openai_short.jsonl`

konumuna koy, sonra haber ver -- `data/interim/{ai_raw_short,ai_humanized_short}/`'e birlestirip (yedek alarak) `build_features.py` + yeniden egitim adimlarina gecerim.
